# Gold Layer

In [0]:
silver_df = spark.table("learntrack_lms_analytics.default.silver_lms")

silver_df.show(10)

+---------+----------+------------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+-----------------+--------------------+------------+----------+-----------------+-----------------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+
|course_id|learner_id|enrolment_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|     learner_name|               email|phone_number|      city|registration_date|subscription_type|        course_title|          category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|
+---------+----------+------------+----------+------------------------+-----------------

## Gold Table 1 — Course Completion Rate
Business Question

What percentage of learners completed each course?

## Step 1: Aggregate

In [0]:
from pyspark.sql.functions import *

course_completion = (
    silver_df
    .groupBy(
        "course_id",
        "course_title",
        "category"
    )
    .agg(
        count("*").alias("total_enrolments"),

        sum(
            when(col("status") == "Completed", 1).otherwise(0)
        ).alias("completed_learners")
    )
)

## Step 2: Calculate Completion %

In [0]:
course_completion = course_completion.withColumn(
    "completion_rate",
    round(
        col("completed_learners") * 100 / col("total_enrolments"),
        2
    )
)

## Step 3: Classify Courses

In [0]:
course_completion = course_completion.withColumn(
    "completion_category",
    when(col("completion_rate") >= 80, "High Completion")
    .when(col("completion_rate") >= 50, "Moderate")
    .otherwise("At Risk")
)

In [0]:
course_completion.show(10)

+---------+--------------------+------------------+----------------+------------------+---------------+-------------------+
|course_id|        course_title|          category|total_enrolments|completed_learners|completion_rate|completion_category|
+---------+--------------------+------------------+----------------+------------------+---------------+-------------------+
|   CRS001|Python for Data S...|      Data Science|              31|                17|          54.84|           Moderate|
|   CRS012|Motion Graphics &...|            Design|              32|                11|          34.38|            At Risk|
|   CRS006|  HTML & CSS Mastery|   Web Development|              48|                12|           25.0|            At Risk|
|   CRS015|Azure Data Engine...|   Cloud Computing|              28|                12|          42.86|            At Risk|
|   CRS036| Advanced Python OOP|      Data Science|              22|                 7|          31.82|            At Risk|
|   CRS0

In [0]:
course_completion.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.gold_course_completion")

## Gold Table 2 — Learner Engagement
Business Question

Which learners are active and which are inactive?

In [0]:
learner_engagement = silver_df.withColumn(
    "days_since_last_activity",
    datediff(current_date(), col("last_activity_date"))
)

## Engagement Status

In [0]:
learner_engagement = learner_engagement.withColumn(
    "engagement_status",
    when(col("days_since_last_activity") <= 7, "Highly Active")
    .when(col("days_since_last_activity") <= 30, "Active")
    .otherwise("Disengaged")
)

In [0]:
learner_engagement.show(10)

+---------+----------+------------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+-----------------+--------------------+------------+----------+-----------------+-----------------+--------------------+------------------+-------------+---------------+--------------+----------------+---------+------------------------+-----------------+
|course_id|learner_id|enrolment_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|     learner_name|               email|phone_number|      city|registration_date|subscription_type|        course_title|          category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|days_since_last_activity|engagement_status|
+-

In [0]:
learner_engagement.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.gold_learner_engagement")

## Gold Table 3 — Instructor Performance
Business Question

Which instructors perform best?

In [0]:
instructor_performance = (
    silver_df
    .groupBy(
        "instructor_id",
        "instructor_name"
    )
    .agg(

        avg("assessment_score").alias("avg_score"),

        avg("feedback_rating").alias("avg_rating"),

        count("*").alias("total_enrolments"),

        sum(
            when(col("status")=="Completed",1).otherwise(0)
        ).alias("completed")
    )
)

## Completion Rate

In [0]:
instructor_performance = instructor_performance.withColumn(
    "completion_rate",
    round(
        col("completed")*100/col("total_enrolments"),
        2
    )
)

In [0]:
instructor_performance.show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+---------------+-----------------+------------------+----------------+---------+---------------+----+
|instructor_id|instructor_name|        avg_score|        avg_rating|total_enrolments|completed|completion_rate|rank|
+-------------+---------------+-----------------+------------------+----------------+---------+---------------+----+
|       INS013|        Unknown|73.35411764705883|3.1578947368421053|              33|       17|          51.52|   1|
|       INS009|   Suresh Gupta|74.00602409638557| 2.870967741935484|             170|       83|          48.82|   2|
|       INS001|   Rajiv Sharma|72.17545454545456|3.4166666666666665|              23|       11|          47.83|   3|
|       INS010|    Anjali Iyer|68.97631578947369| 3.142857142857143|             120|       57|           47.5|   4|
|       INS002|     Meera Nair|70.02799999999998|            3.1125|             159|       75|          47.17|   5|
|       INS006|        Unknown|66.38722222222222| 3.263157894736

## Rank Instructors

In [0]:
from pyspark.sql.window import Window

windowSpec = Window.orderBy(desc("completion_rate"))

instructor_performance = instructor_performance.withColumn(
    "rank",
    dense_rank().over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
instructor_performance.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.gold_instructor_performance")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Gold Table 4 — Assessment Performance
Business Question

Which assessments are difficult?

In [0]:
assessment_performance = (
    silver_df
    .groupBy(
        "course_id",
        "course_title"
    )
    .agg(

        avg("assessment_score").alias("average_score"),

        max("assessment_score").alias("highest_score"),

        min("assessment_score").alias("lowest_score"),

        count("assessment_score").alias("attempts")
    )
)

Pass Rate

Let's assume 40 marks is the passing score.

In [0]:
assessment_performance = (
    silver_df
    .groupBy(
        "course_id",
        "course_title"
    )
    .agg(

        avg("assessment_score").alias("average_score"),

        sum(
            when(col("assessment_score") >= 40,1)
            .otherwise(0)
        ).alias("passed"),

        count("assessment_score").alias("attempted")
    )
)

## Calculate Pass %

In [0]:
assessment_performance = assessment_performance.withColumn(
    "pass_rate",
    round(
        col("passed")*100/col("attempted"),
        2
    )
)

In [0]:
assessment_performance.show(10)

+---------+--------------------+-----------------+------+---------+---------+
|course_id|        course_title|    average_score|passed|attempted|pass_rate|
+---------+--------------------+-----------------+------+---------+---------+
|   CRS001|Python for Data S...|74.32705882352941|    17|       17|    100.0|
|   CRS012|Motion Graphics &...|66.32727272727273|    11|       11|    100.0|
|   CRS006|  HTML & CSS Mastery|77.97666666666666|    12|       12|    100.0|
|   CRS015|Azure Data Engine...|69.07249999999999|    12|       12|    100.0|
|   CRS036| Advanced Python OOP|74.94571428571429|     7|        7|    100.0|
|   CRS007|Business Analytic...|68.67909090909092|    11|       11|    100.0|
|   CRS052|  Svelte & SvelteKit|        69.306875|    16|       16|    100.0|
|   CRS018|Flutter App Devel...|72.74666666666666|    12|       12|    100.0|
|   CRS040| TypeScript Advanced|73.03533333333334|    15|       15|    100.0|
|   CRS060| Bayesian Statistics|66.38722222222222|    18|       

In [0]:
assessment_performance.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.gold_assessment_performance")

## Gold Table 5 — Dropout & Re-enrolment
Business Question

Who dropped and enrolled again?

In [0]:
reenrolment = silver_df.filter(
    col("attempts") >= 2
)

In [0]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy(
    "learner_id",
    "course_id"
).orderBy(desc("enrol_date"))

In [0]:
latest_attempt = silver_df.withColumn(
    "row_num",
    row_number().over(windowSpec)
)

## Keep latest

In [0]:
latest_attempt = latest_attempt.filter(
    col("row_num")==1
)

In [0]:
latest_attempt.show(10)

+---------+----------+------------+----------+------------------------+----------------------+-----------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+-------------+--------------------+------------+-------+-----------------+-----------------+--------------------+---------------+-------------+---------------+--------------+----------------+---------+-------+
|course_id|learner_id|enrolment_id|enrol_date|expected_completion_date|actual_completion_date|     status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days| learner_name|               email|phone_number|   city|registration_date|subscription_type|        course_title|       category|instructor_id|instructor_name|duration_hours|difficulty_level|price_inr|row_num|
+---------+----------+------------+----------+------------------------+---------------------

In [0]:
latest_attempt.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("learntrack_lms_analytics.default.gold_dropout_reenrolment")

| Table                         | Purpose                             |
| ----------------------------- | ----------------------------------- |
| `gold_course_completion`      | Course-wise completion rates        |
| `gold_learner_engagement`     | Active vs disengaged learners       |
| `gold_instructor_performance` | Instructor rankings and metrics     |
| `gold_assessment_performance` | Assessment pass rates and scores    |
| `gold_dropout_reenrolment`    | Latest enrolments and re-enrolments |
